# OMO Ecological Informatics

## Notebook 02 – Exploratory Ecological Analysis

### Objective

This notebook explores the standardized tree inventory database to understand species composition, abundance patterns, habitat differences, and management zone characteristics.

The analyses provide ecological insight and identify data issues that should be resolved before machine learning.

## Import Required Libraries

This notebook uses pandas for data manipulation and matplotlib for exploratory visualization.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
# Project paths

project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"

## Load the Master Database

The standardized master database generated in Notebook 01 is used as the input for all analyses in this notebook.

In [3]:
master_file = processed_path / "tree_master_database_v1.0.csv"

tree = pd.read_csv(master_file)

tree.head()

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
0,Alstonei boonei,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,2,2,0.0027,1.1560,2.0400,1.5984,Buffer,Major River
1,Blighia sapida,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,1.0,5,5,0.0068,2.8901,5.1020,3.9961,Buffer,Major River
2,Bombax buonopozene,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,NaN,5,7,0.0096,4.0462,5.1020,4.5741,Buffer,Major River
3,Bosqueia angolensis,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN,3,4,0.0055,2.3121,3.0612,2.6866,Buffer,Major River
4,Celtis zenkeri,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,NaN,4,5,0.0068,2.8901,4.0816,3.4859,Buffer,Major River


## Dataset Summary

The first step is to summarize the dimensions and basic characteristics of the master database.

In [4]:
print("="*60)
print("MASTER DATABASE SUMMARY")
print("="*60)

print(f"Rows      : {tree.shape[0]}")
print(f"Columns   : {tree.shape[1]}")

print(f"\nUnique Species : {tree['Species'].nunique()}")

print(f"\nZones")
print(tree["Zone"].value_counts())

print(f"\nHabitats")
print(tree["Habitat"].value_counts())

MASTER DATABASE SUMMARY
Rows      : 222
Columns   : 19

Unique Species : 141

Zones
Zone
Buffer        87
Core          73
Transition    62
Name: count, dtype: int64

Habitats
Habitat
Major River    78
Stream         73
Upland         71
Name: count, dtype: int64


In [5]:
tree["Species"].nunique()

141

In [6]:
tree["Species"].sort_values().tolist()

['14',
 '18',
 '24',
 '24',
 '25',
 '27',
 '28',
 '33',
 'Afzelia africana',
 'Albizia auxilary',
 'Albizia ferruginea',
 'Albizia zygia',
 'Albizia zygia',
 'Alstonei bonei',
 'Alstonei boonei',
 'Alstonei boonei',
 'Alstonia boonei',
 'Alstonia boonei',
 'Amphimas tetracoides',
 'Aningeria robusta',
 'Annonidium manni',
 'Annonidium manni',
 'Annonidium manni',
 'Anthocleista djalonensis',
 'Anthocliesta vogeli',
 'Anthocliesta vogelii',
 'Anthonotha macrophylla',
 'Artocarpus communis',
 'Artocarpus communis',
 'Bambusa vulgaris',
 'Bambusa vulgaris',
 'Baphia nitida',
 'Bligha sapida',
 'Blighia Sapida',
 'Blighia sapida',
 'Blighia sapida',
 'Blighia sapida',
 'Blighia sapida',
 'Blighia unijigata',
 'Bombax buonopozene',
 'Bombax buonopozene',
 'Bombax buonopozene',
 'Bombax buonopozense',
 'Bosquei angolensis',
 'Bosqueia angolensis',
 'Brachystegia eurycoma',
 'Bridelia ferruginea',
 'Bridelia ferruginea',
 'Carica papaya',
 'Cassia saimen',
 'Ceiba pentandra',
 'Ceiba pentandr

## Remove Invalid Species Records

The master database is inspected for records that do not represent valid species names.

Examples include:

- Numeric values
- Empty records
- Placeholder entries

These records are removed before taxonomic harmonization.

In [7]:
# Remove numeric species names

tree = tree[
    ~tree["Species"]
      .astype(str)
      .str.fullmatch(r"\d+")
].copy()

tree.reset_index(drop=True, inplace=True)

print(f"Remaining records: {tree.shape[0]}")

Remaining records: 214


In [8]:
print(f"Unique species: {tree['Species'].nunique()}")

Unique species: 134


## Standardize Capitalization

Species names should follow a consistent capitalization style.

This step standardizes capitalization and removes unnecessary whitespace while preserving the original spelling.

In [9]:
tree["Species"] = (
    tree["Species"]
        .str.strip()
        .str.title()
)

In [10]:
species = sorted(tree["Species"].unique())

print(f"Unique species after formatting: {len(species)}")

Unique species after formatting: 131


## Detect Potential Taxonomic Inconsistencies

Species names that differ by only a few characters may represent spelling inconsistencies.

Python's fuzzy string matching is used to identify candidate names for manual review.

In [11]:
from difflib import get_close_matches

In [12]:
possible_duplicates = []

for sp in species:

    matches = get_close_matches(
        sp,
        species,
        n=5,
        cutoff=0.90
    )

    for match in matches:

        if sp != match:

            possible_duplicates.append(
                (sp, match)
            )

In [13]:
duplicates = (
    pd.DataFrame(
        possible_duplicates,
        columns=["Species_1","Species_2"]
    )
    .drop_duplicates()
)

duplicates

,Species_1,Species_2
0,Alstonei Bonei,Alstonei Boonei
1,Alstonei Boonei,Alstonei Bonei
2,Alstonei Boonei,Alstonia Boonei
3,Alstonia Boonei,Alstonei Boonei
4,Anthocliesta Vogeli,Anthocliesta Vogelii
...,...,...
83,Terminalia Superba,Termialia Superba
84,Uapaca Heudeloti,Uapaca Heudelotii
85,Uapaca Heudelotii,Uapaca Heudeloti
86,Zanthoxylum Xanthoxyloides,Zanthoxylum Zanthoxyloides


## Generate Taxonomic Review Table

Potential spelling inconsistencies identified during fuzzy matching are exported for manual review.

Each suggested correction will be examined before it is applied to the master database.

In [14]:
# Remove duplicate pairs

duplicates = duplicates.sort_values(
    ["Species_1", "Species_2"]
).drop_duplicates()

duplicates.reset_index(drop=True, inplace=True)

duplicates.head(20)

,Species_1,Species_2
0,Alstonei Bonei,Alstonei Boonei
1,Alstonei Boonei,Alstonei Bonei
2,Alstonei Boonei,Alstonia Boonei
3,Alstonia Boonei,Alstonei Boonei
4,Anthocliesta Vogeli,Anthocliesta Vogelii
5,Anthocliesta Vogelii,Anthocliesta Vogeli
6,Bligha Sapida,Blighia Sapida
7,Blighia Sapida,Bligha Sapida
8,Bombax Buonopozene,Bombax Buonopozense
9,Bombax Buonopozense,Bombax Buonopozene


In [15]:
# Remove duplicate pairs

duplicates = duplicates.sort_values(
    ["Species_1", "Species_2"]
).drop_duplicates()

duplicates.reset_index(drop=True, inplace=True)

duplicates.head(20)

,Species_1,Species_2
0,Alstonei Bonei,Alstonei Boonei
1,Alstonei Boonei,Alstonei Bonei
2,Alstonei Boonei,Alstonia Boonei
3,Alstonia Boonei,Alstonei Boonei
4,Anthocliesta Vogeli,Anthocliesta Vogelii
5,Anthocliesta Vogelii,Anthocliesta Vogeli
6,Bligha Sapida,Blighia Sapida
7,Blighia Sapida,Bligha Sapida
8,Bombax Buonopozene,Bombax Buonopozense
9,Bombax Buonopozense,Bombax Buonopozene


In [16]:
duplicates["Approved_Name"] = ""

duplicates["Status"] = "Pending"

duplicates

,Species_1,Species_2,Approved_Name,Status
0,Alstonei Bonei,Alstonei Boonei,,Pending
1,Alstonei Boonei,Alstonei Bonei,,Pending
2,Alstonei Boonei,Alstonia Boonei,,Pending
3,Alstonia Boonei,Alstonei Boonei,,Pending
4,Anthocliesta Vogeli,Anthocliesta Vogelii,,Pending
...,...,...,...,...
83,Terminalia Superba,Termialia Superba,,Pending
84,Uapaca Heudeloti,Uapaca Heudelotii,,Pending
85,Uapaca Heudelotii,Uapaca Heudeloti,,Pending
86,Zanthoxylum Xanthoxyloides,Zanthoxylum Zanthoxyloides,,Pending


In [17]:
review_file = processed_path / "species_review_table.csv"

duplicates.to_csv(
    review_file,
    index=False
)

print(review_file)

C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\data\processed\species_review_table.csv


## Standardize Botanical Name Formatting

Scientific names are formatted according to botanical convention:

- Genus begins with a capital letter.
- Specific epithet is written in lowercase.

This improves consistency without altering taxonomy.

In [18]:
def botanical_case(name):
    """
    Format scientific names using botanical conventions.
    """

    if pd.isna(name):
        return name

    parts = str(name).strip().split()

    if len(parts) == 1:
        return parts[0].capitalize()

    return parts[0].capitalize() + " " + " ".join(
        p.lower() for p in parts[1:]
    )

In [19]:
tree["Species"] = tree["Species"].apply(botanical_case)

tree["Species"].head(20)

0               Alstonei boonei
1                Blighia sapida
2            Bombax buonopozene
3           Bosqueia angolensis
4                Celtis zenkeri
5          Cleistopholis patens
6            Diallium guineense
7               Dillenia indica
8               Diospyros dendo
9              Ficus exasperata
10             Grewia pubescens
11             Homalium aylmeri
12     Lecaniodiscus cupaniodes
13        Margartaria discoidea
14         Musanga cecropioides
15    Nesogordonia papaverifera
16           Pavetta corymbosia
17             Picralima nitida
18         Pycanthus angolensis
19      Ricinodendron heudeotii
Name: Species, dtype: object

## Compute Taxonomic Similarity Scores

Potential duplicate species names are assigned similarity scores using
Python's SequenceMatcher.

Higher scores indicate a greater likelihood that two names represent
the same species recorded with different spellings.

In [20]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

duplicates["Similarity"] = duplicates.apply(
    lambda x: similarity(x["Species_1"], x["Species_2"]),
    axis=1
)

duplicates = duplicates.sort_values(
    "Similarity",
    ascending=False
)

duplicates.head(30)

,Species_1,Species_2,Approved_Name,Status,Similarity
25,Lecaniodiscus Cupanioides,Lecaniodiscus Cupaniodes,,Pending,0.979592
24,Lecaniodiscus Cupaniodes,Lecaniodiscus Cupanioides,,Pending,0.979592
62,Ricinodendron Heudeloti,Ricinodendron Heudelotii,,Pending,0.978723
69,Ricinodendron Heudeotii,Ricinodendron Heudelotii,,Pending,0.978723
66,Ricinodendron Heudelotii,Ricinodendron Heudeotii,,Pending,0.978723
65,Ricinodendron Heudelotii,Ricinodendron Heudeloti,,Pending,0.978723
28,Margaritaria Discoidea,Margartaria Discoidea,,Pending,0.976744
29,Margartaria Discoidea,Margaritaria Discoidea,,Pending,0.976744
12,Chrysophyllum Albidum,Chrysophylum Albidum,,Pending,0.975610
13,Chrysophylum Albidum,Chrysophyllum Albidum,,Pending,0.975610


## Species Frequency

Species occurrence frequencies are calculated to support taxonomic review.

Frequently occurring spellings often represent the preferred form,
although all corrections will be confirmed using botanical references.

In [21]:
species_frequency = (
    tree["Species"]
        .value_counts()
        .reset_index()
)

species_frequency.columns = ["Species", "Occurrences"]

species_frequency.head(20)

,Species,Occurrences
0,Diospyros dendo,6
1,Sterculia rhinopetala,6
2,Blighia sapida,5
3,Cleistopholis patens,5
4,Celtis zenkeri,5
5,Terminalia superba,4
6,Ficus exasperata,4
7,Trichilia monadelpha,4
8,Funtumia elastica,4
9,Strombosia pustulata,4


In [22]:
duplicates = duplicates.merge(
    species_frequency,
    left_on="Species_1",
    right_on="Species",
    how="left"
).rename(columns={"Occurrences":"Occurrences_1"})

duplicates.drop(columns="Species", inplace=True)

duplicates = duplicates.merge(
    species_frequency,
    left_on="Species_2",
    right_on="Species",
    how="left"
).rename(columns={"Occurrences":"Occurrences_2"})

duplicates.drop(columns="Species", inplace=True)

In [23]:
duplicates.head(20)

,Species_1,Species_2,Approved_Name,Status,Similarity,Occurrences_1,Occurrences_2
0,Lecaniodiscus Cupanioides,Lecaniodiscus Cupaniodes,,Pending,0.979592,NaN,NaN
1,Lecaniodiscus Cupaniodes,Lecaniodiscus Cupanioides,,Pending,0.979592,NaN,NaN
2,Ricinodendron Heudeloti,Ricinodendron Heudelotii,,Pending,0.978723,NaN,NaN
3,Ricinodendron Heudeotii,Ricinodendron Heudelotii,,Pending,0.978723,NaN,NaN
4,Ricinodendron Heudelotii,Ricinodendron Heudeotii,,Pending,0.978723,NaN,NaN
5,Ricinodendron Heudelotii,Ricinodendron Heudeloti,,Pending,0.978723,NaN,NaN
6,Margaritaria Discoidea,Margartaria Discoidea,,Pending,0.976744,NaN,NaN
7,Margartaria Discoidea,Margaritaria Discoidea,,Pending,0.976744,NaN,NaN
8,Chrysophyllum Albidum,Chrysophylum Albidum,,Pending,0.975610,NaN,NaN
9,Chrysophylum Albidum,Chrysophyllum Albidum,,Pending,0.975610,NaN,NaN


In [24]:
duplicates["Suggested_Name"] = duplicates.apply(
    lambda x:
        x["Species_1"]
        if x["Occurrences_1"] >= x["Occurrences_2"]
        else x["Species_2"],
    axis=1
)

In [25]:
duplicates = duplicates[
    [
        "Species_1",
        "Species_2",
        "Similarity",
        "Occurrences_1",
        "Occurrences_2",
        "Suggested_Name",
        "Approved_Name",
        "Status"
    ]
]

duplicates.head(20)

,Species_1,Species_2,Similarity,Occurrences_1,Occurrences_2,Suggested_Name,Approved_Name,Status
0,Lecaniodiscus Cupanioides,Lecaniodiscus Cupaniodes,0.979592,NaN,NaN,Lecaniodiscus Cupaniodes,,Pending
1,Lecaniodiscus Cupaniodes,Lecaniodiscus Cupanioides,0.979592,NaN,NaN,Lecaniodiscus Cupanioides,,Pending
2,Ricinodendron Heudeloti,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending
3,Ricinodendron Heudeotii,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending
4,Ricinodendron Heudelotii,Ricinodendron Heudeotii,0.978723,NaN,NaN,Ricinodendron Heudeotii,,Pending
5,Ricinodendron Heudelotii,Ricinodendron Heudeloti,0.978723,NaN,NaN,Ricinodendron Heudeloti,,Pending
6,Margaritaria Discoidea,Margartaria Discoidea,0.976744,NaN,NaN,Margartaria Discoidea,,Pending
7,Margartaria Discoidea,Margaritaria Discoidea,0.976744,NaN,NaN,Margaritaria Discoidea,,Pending
8,Chrysophyllum Albidum,Chrysophylum Albidum,0.975610,NaN,NaN,Chrysophylum Albidum,,Pending
9,Chrysophylum Albidum,Chrysophyllum Albidum,0.975610,NaN,NaN,Chrysophyllum Albidum,,Pending


## Export Taxonomic Review Table

The review table is exported for manual verification.

Corrections will only be applied after expert review.

In [26]:
review_file = processed_path / "species_review_table.csv"

duplicates.to_csv(review_file, index=False)

print(f"Review table saved to:\n{review_file}")

Review table saved to:
C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\data\processed\species_review_table.csv


## Select High-Confidence Candidate Matches

Only candidate pairs with high similarity scores are retained for manual review.

A similarity threshold of 0.95 is used to prioritize likely spelling variants while minimizing false positives.

In [27]:
high_confidence = duplicates[
    duplicates["Similarity"] >= 0.95
].copy()

print(f"High-confidence matches: {len(high_confidence)}")

high_confidence

High-confidence matches: 62


,Species_1,Species_2,Similarity,Occurrences_1,Occurrences_2,Suggested_Name,Approved_Name,Status
0,Lecaniodiscus Cupanioides,Lecaniodiscus Cupaniodes,0.979592,NaN,NaN,Lecaniodiscus Cupaniodes,,Pending
1,Lecaniodiscus Cupaniodes,Lecaniodiscus Cupanioides,0.979592,NaN,NaN,Lecaniodiscus Cupanioides,,Pending
2,Ricinodendron Heudeloti,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending
3,Ricinodendron Heudeotii,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending
4,Ricinodendron Heudelotii,Ricinodendron Heudeotii,0.978723,NaN,NaN,Ricinodendron Heudeotii,,Pending
...,...,...,...,...,...,...,...,...
57,Ricinodendron Heudeloti,Ricinodendron Heudeotii,0.956522,NaN,NaN,Ricinodendron Heudeotii,,Pending
58,Strombosia Postulata,Strombosia Pustulata,0.950000,NaN,NaN,Strombosia Pustulata,,Pending
59,Strombosia Pustulata,Strombosia Postulata,0.950000,NaN,NaN,Strombosia Postulata,,Pending
60,Musanga Cecropoidies,Musanga Cecropioides,0.950000,NaN,NaN,Musanga Cecropioides,,Pending


In [28]:
high_confidence["Decision"] = ""

high_confidence.head()

,Species_1,Species_2,Similarity,Occurrences_1,Occurrences_2,Suggested_Name,Approved_Name,Status,Decision
0,Lecaniodiscus Cupanioides,Lecaniodiscus Cupaniodes,0.979592,NaN,NaN,Lecaniodiscus Cupaniodes,,Pending,
1,Lecaniodiscus Cupaniodes,Lecaniodiscus Cupanioides,0.979592,NaN,NaN,Lecaniodiscus Cupanioides,,Pending,
2,Ricinodendron Heudeloti,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending,
3,Ricinodendron Heudeotii,Ricinodendron Heudelotii,0.978723,NaN,NaN,Ricinodendron Heudelotii,,Pending,
4,Ricinodendron Heudelotii,Ricinodendron Heudeotii,0.978723,NaN,NaN,Ricinodendron Heudeotii,,Pending,


In [29]:
review_candidates = (
    processed_path /
    "high_confidence_species_review.csv"
)

high_confidence.to_csv(
    review_candidates,
    index=False
)

print(review_candidates)

C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\data\processed\high_confidence_species_review.csv


## Create Species Dictionary Template

A species dictionary is created to map original names to accepted scientific names.

This dictionary provides a transparent and reusable record of all taxonomic corrections.

In [31]:
species_dictionary = tree[
    ["Species"]
].drop_duplicates().copy()

species_dictionary = species_dictionary.rename(
    columns={"Species": "Original_Name"}
)

species_dictionary["Accepted_Name"] = ""

species_dictionary["Family"] = ""

species_dictionary["Taxonomic_Status"] = ""

species_dictionary.head()

,Original_Name,Accepted_Name,Family,Taxonomic_Status
0,Alstonei boonei,,,
1,Blighia sapida,,,
2,Bombax buonopozene,,,
3,Bosqueia angolensis,,,
4,Celtis zenkeri,,,


In [32]:
species_dictionary["Accepted_Name"] = ""

species_dictionary["Family"] = ""

species_dictionary["Genus"] = ""

species_dictionary["Taxonomic_Status"] = ""

species_dictionary["Remarks"] = ""

species_dictionary.head()

,Original_Name,Accepted_Name,Family,Taxonomic_Status,Genus,Remarks
0,Alstonei boonei,,,,,
1,Blighia sapida,,,,,
2,Bombax buonopozene,,,,,
3,Bosqueia angolensis,,,,,
4,Celtis zenkeri,,,,,


In [33]:
dictionary_template = (
    processed_path /
    "species_dictionary_template.csv"
)

species_dictionary.to_csv(
    dictionary_template,
    index=False
)

print(f"Template saved to:\n{dictionary_template}")

Template saved to:
C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\data\processed\species_dictionary_template.csv


In [34]:
pd.read_csv(dictionary_template).head()

,Original_Name,Accepted_Name,Family,Taxonomic_Status,Genus,Remarks
0,Alstonei boonei,NaN,NaN,NaN,NaN,NaN
1,Blighia sapida,NaN,NaN,NaN,NaN,NaN
2,Bombax buonopozene,NaN,NaN,NaN,NaN,NaN
3,Bosqueia angolensis,NaN,NaN,NaN,NaN,NaN
4,Celtis zenkeri,NaN,NaN,NaN,NaN,NaN


# Notebook Summary

## Completed Tasks

✔ Loaded the standardized master database

✔ Removed invalid species records

✔ Standardized scientific name formatting

✔ Identified potential duplicate species names

✔ Generated taxonomic review table

✔ Created species dictionary template

---

## Outputs Generated

- species_review_table.csv
- species_dictionary_template.csv

---

## Next Notebook

Notebook 03 – Taxonomic Harmonization